# Machine Learning System Design: Practical Real-World Case Studies

This module features 11 end-to-end production case studies covering planetary-scale AI/ML systems:
1. **Personalized Recommendation System**: Multi-stage cascade (Two-Tower retrieval + DLRM ranking + MMR diversity).
2. **Real-Time Payment Fraud Detection**: Low-latency GBDT + rules engine with sub-30ms P99 SLAs.
3. **Hybrid Web Search Engine**: Lexical (BM25) and dense semantic vector search with cross-encoder neural reranking.
4. **Enterprise Image Classification Platform**: High-throughput Vision Transformer pipeline with TensorRT.
5. **Real-Time Video Object Detection**: Low-latency edge perception (YOLOv8 + ByteTrack).
6. **Enterprise RAG System**: Access-controlled private document retrieval with citation validation.
7. **Enterprise Customer Support Chatbot**: Multi-turn agent with state machines and tool calling.
8. **Intelligent Document Processing (IDP)**: Multimodal layout transformers and OCR math reconciliation.
9. **High-Throughput LLM Inference Service**: vLLM serving cluster with PagedAttention and speculative decoding.
10. **Autonomous AI Agent Platform**: Sandboxed code execution, durable state checkpointing, and MCP tools.
11. **Multimodal Content Moderation Platform**: Defense-in-depth safety scanning (PhotoDNA + multi-modal classifiers).

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('08-system-design/case-studies/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import case_study_sim
print('case_study_sim loaded')

## 1. Multi-Stage Recommendation Engine Simulation
Simulating Two-Tower candidate retrieval followed by dense cosine + popularity fine ranking.

In [ ]:
from case_study_sim import CatalogItem, RecommendationSystemEngine

catalog = [
    CatalogItem("video_101", ["action", "sci-fi"], popularity=0.85, embedding=[0.9, 0.2]),
    CatalogItem("video_102", ["comedy", "romance"], popularity=0.60, embedding=[0.1, 0.9]),
    CatalogItem("video_103", ["action", "thriller"], popularity=0.92, embedding=[0.85, 0.3]),
    CatalogItem("video_104", ["documentary", "history"], popularity=0.70, embedding=[0.4, 0.4])
]

engine = RecommendationSystemEngine(catalog)

# Step 1: Candidate Generation (User interested in action)
candidates = engine.candidate_generation(["action"], top_k=3)
print("Stage 1 (Retrieval): Candidates selected:", [c.item_id for c in candidates])

# Step 2: Fine-Ranking with User Embedding [1.0, 0.0]
user_vector = [1.0, 0.0]
ranked_results = engine.rank_candidates(user_vector, candidates)
print("\nStage 2 (Fine Ranking):")
for item, score in ranked_results:
    print(f" -> {item.item_id:10s} (Tags: {', '.join(item.tags)}): Score={score:.4f}")

## 2. Real-Time Transaction Fraud Detection Simulation
Combining hard regulatory sanctions rules with an optimized logistic risk scoring model.

In [ ]:
from case_study_sim import Transaction, FraudDetectionEngine

fraud_system = FraudDetectionEngine()

# Test 1: Standard legitimate domestic purchase
tx_legit = Transaction(txn_id="tx_01", user_id="u_101", amount=35.0, country="US", is_foreign=False, velocity_1h=1)
res_legit = fraud_system.evaluate_transaction(tx_legit)
print(f"Transaction 1 (Legitimate): Action={res_legit['action']}, Risk Score={res_legit['risk_score']:.4f}")

# Test 2: Rapid foreign transactions (Velocity spike)
tx_spike = Transaction(txn_id="tx_02", user_id="u_102", amount=850.0, country="US", is_foreign=True, velocity_1h=9)
res_spike = fraud_system.evaluate_transaction(tx_spike)
print(f"Transaction 2 (Suspicious): Action={res_spike['action']}, Risk Score={res_spike['risk_score']:.4f} (Reason: {res_spike['reason']})")

# Test 3: Transaction from sanctioned/high-risk jurisdiction
tx_sanction = Transaction(txn_id="tx_03", user_id="u_103", amount=12.0, country="XX", is_foreign=True, velocity_1h=1)
res_sanction = fraud_system.evaluate_transaction(tx_sanction)
print(f"Transaction 3 (Sanctioned): Action={res_sanction['action']}, Risk Score={res_sanction['risk_score']:.4f} (Reason: {res_sanction['reason']})")

## 3. Hybrid Search Engine Simulation
Combining BM25 keyword matching with dense semantic embeddings using score fusion.

In [ ]:
from case_study_sim import HybridSearchEngine

docs = {
    "doc_deep_learning": "deep learning architectures rely on backpropagation and gradient descent",
    "doc_transformers": "the transformer architecture uses multi head self attention mechanisms",
    "doc_sql_databases": "relational databases use structured query language for transactions"
}

doc_embeddings = {
    "doc_deep_learning": [0.8, 0.4, 0.1],
    "doc_transformers": [0.9, 0.8, 0.0],
    "doc_sql_databases": [0.1, 0.1, 0.9]
}

search_engine = HybridSearchEngine(docs, doc_embeddings)

# Query: "transformer attention"
query_terms = ["transformer", "attention"]
query_vector = [0.95, 0.85, 0.0]

search_results = search_engine.search(query_terms, query_vector, alpha=0.6, top_k=2)
print("Hybrid Search Results for 'transformer attention':")
for doc_id, score in search_results:
    print(f" - [{doc_id:18s}] Combined Score={score:.4f}: '{docs[doc_id][:55]}...'")